<a href="https://colab.research.google.com/github/SougataJoy/IO-List-Creation/blob/main/IO_List_Updated005.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ================  IO List Generator - Optimized v2  ================
# ---- Imports ----
import ipywidgets as widgets
from IPython.display import display
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from openpyxl.drawing.image import Image
from openpyxl.utils import get_column_letter, range_boundaries
from google.colab import files
from datetime import datetime
import pytz
import os

# ---- Colab cleanup ----
for f in os.listdir():
    if f.endswith(".png") or f.endswith(".xlsx"):
        os.remove(f)

# ---- Timezone & date ----
ist = pytz.timezone('Asia/Kolkata')
today_date      = datetime.now(ist).strftime("%Y-%b-%d_%H-%M")
today_date_only = datetime.now(ist).strftime("%d-%m-%Y")

# ---- Reusable style constants ----
THIN  = Side(style='thin')
THICK = Side(style='thick')

YELLOW_FILL = PatternFill(start_color="FFFF00", fill_type="solid")
ORANGE_FILL = PatternFill(start_color="FFC000", fill_type="solid")

TNR_26B    = Font(name='Times New Roman', size=26, bold=True)
TNR_9B     = Font(name='Times New Roman', size=9,  bold=True)
CAL_18B    = Font(name='Calibri', size=18, bold=True)
CAL_16B    = Font(name='Calibri', size=16, bold=True)
CAL_16     = Font(name='Calibri', size=16, bold=False)

CENTER     = Alignment(horizontal='center', vertical='center', wrap_text=True)
CENTER_H   = Alignment(horizontal='center', wrap_text=True)
RIGHT      = Alignment(horizontal='right', wrap_text=True)
LEFT       = Alignment(horizontal='left', wrap_text=True)

def set_cell1(ws, cell, value, merge_with=None):
    """
    Set a cell's value with centered, wrapped alignment.
    Optionally merge it with another cell (e.g., merge_with='O6').
    """
    if merge_with:
        ws.merge_cells(f'{cell}:{merge_with}')
    hdr_font  = TNR_9B
    hdr_align = CENTER
    ws[cell].value     = value
    ws[cell].font      = hdr_font
    ws[cell].alignment = Alignment(
        horizontal='center',
        vertical='center',
        wrap_text=True
    )

class Node:
    def __init__(self, number):
        if number < 10:
            self.number = f"0{number}"
        else:
            self.number = f"{number}"

class Card:
    def __init__(self, number):
        if number < 10:
            self.number = f"N0{number}"
        else:
            self.number = f"N{number}"
    def Card(self):
      while True:
          try:
              c_f = 1
              c_l = int(input('Enter Last Card Number: '))
              # Check negative
              if c_l > 12:
                  print("Card Qty cannot gt 12.")
                  continue
              # Check last Card
              if c_l < c_f:
                  print("Last Card number cannot be less than 1 number.")
                  continue
              break
          except ValueError:
              print("Invalid input. Please enter integers only.")
      while c_f <= c_l:
        card = Card(c_f)
        print(card.number,'_',card.number)
        c_f += 1


# ---- Project-level globals (filled from user input) ----
Client_Name    = ""
Project_Name   = ""
po_reference_no = ""
jc_no          = ""


# ==============================================================
#  Utility helpers
# ==============================================================

def apply_border(ws, cell_range, side_style='thin'):
    """Apply outer border of given style to every edge cell in range."""
    s = Side(style=side_style)
    min_col, min_row, max_col, max_row = range_boundaries(cell_range)
    for row in range(min_row, max_row + 1):
        for col in range(min_col, max_col + 1):
            b = ws.cell(row=row, column=col).border
            ws.cell(row=row, column=col).border = Border(
                top    = s if row == min_row else b.top,
                bottom = s if row == max_row else b.bottom,
                left   = s if col == min_col else b.left,
                right  = s if col == max_col else b.right,
            )

def merge(ws, start_col, end_col, row):
    ws.merge_cells(start_row=row, start_column=start_col,
                   end_row=row,   end_column=end_col)

def set_cell(ws, row, col, value=None, font=None, align=None):
    cell = ws.cell(row=row, column=col)
    if value is not None: cell.value = value
    if font:              cell.font  = font
    if align:
      cell.alignment = Alignment(
            horizontal = align.horizontal,
            vertical   = align.vertical,
            wrap_text  = True
      )

def validate_number_int(number):
    try:
        int(number)
        return True
    except (ValueError, TypeError):
        print("Invalid number. Please enter an integer.")
        return False

# ==============================================================
#  IO List sheet builder
# ==============================================================

# IO List header columns – (col_number, header_text or None, width)
IO_COLS = [
    (1,  "Sl. NO",                                     5),
    (2,  "DCS TAG",                                   16),
    (3,  "DCS Tag. No.",                              20),
    (4,  "TAG DESCRIPTION",                           75),
    (5,  "I/O TYPE",                                  10),
    (6,  "Signal Type",                               12),
    (7,  "RANGE",                                      7),   # merged with col 8 for Low/High
    (8,  None,                                         7),
    (9,  "Engg Unit",                                  9),
    (10, "Dcs Node / Slot",                            8),
    (11, "DCS Card Address/Ch. No.",                  12),
    (12, "DCS Card CH. TB.",                          10),
    (13, "Panel No.",                                 24),
    (14, "DCS Wire No.",                              16),
    (15, "Coil SIDE",                                  5),
    (16, "Relay Board (For DO)",                      22),
    (17, "PF SIDE",                                    5),
    (18, "Ferrule at Panel Main TB",                  16),
    (19, "Panel Main TB No.",                         19),
    (20, "Ferrule at Feeder Side (For MCC Panels)",   29),
]

# Headers that occupy full height from row 4 to row 6
SINGLE_COL_HEADERS = {
    1:  "Sl. NO",
    2:  "DCS TAG",
    3:  "DCS Tag. No.",
    4:  "TAG DESCRIPTION",
    5:  "I/O TYPE",
    6:  "Signal Type",
    9:  "Engg Unit",
    10: "Dcs Node / Slot",
    11: "DCS Card Address/Ch. No.",
    12: "DCS Card CH. TB.",
    13: "Panel No.",
    14: "DCS Wire No.",
    18: "Ferrule at Panel Main TB ",
    19: "Panel Main TB No.",
    20: "Ferrule at Feeder Side (For MCC Panels)",
}

# Cols 7-8 merge horizontally → sub-label rows below
# Col 15-18 also split vertically with sub-labels in rows 5-6
# (these match the Sample.xlsx layout)

ROW5_LABELS = {7: "Low",  8: "High"}
ROW6_LABELS = {7: "Min",  8: "Max"}
# Cols 17-18 row5 / row6 sub labels
#ROW5_LABELS.update({17: "Coil Side", 18: "Relay Board (For DO)"})
#ROW6_LABELS.update({17: "",          18: ""})


def add_new_IO_List(wb):
    """Prompt for name, create and format an IO List sheet."""
    while True:
        sheet_name = input("Enter sheet name (blank = 'IO List'): ").strip() or "IO List"
        if sheet_name not in wb.sheetnames:
            break
        print(f"  '{sheet_name}' already exists – try another name.\n")

    ws = wb.create_sheet(title=sheet_name)

    # ---- column widths ----
    for col, _, width in IO_COLS:
        ws.column_dimensions[get_column_letter(col)].width = width

    # ---- rows 1-3: title block ----
    fills   = [YELLOW_FILL, YELLOW_FILL, ORANGE_FILL]
    values  = [
        Client_Name,
        f"{sheet_name} ; {Project_Name}",
        f"{po_reference_no} / {jc_no}",
    ]
    for row in range(1, 4):
        ws[f"A{row}"].fill      = fills[row - 1]
        ws[f"A{row}"].value     = values[row - 1]
        ws[f"A{row}"].font      = TNR_26B
        ws[f"A{row}"].alignment = CENTER
        ws.row_dimensions[row].height = 32
        merge(ws, 1, 20, row)
        apply_border(ws, f"A{row}:T{row}")

    # ---- rows 4-6: column headers ----
    hdr_font  = TNR_9B
    hdr_align = CENTER

    for col, label, _ in IO_COLS:
        cl = get_column_letter(col)
        if col in SINGLE_COL_HEADERS:
            ws.merge_cells(start_row=4, start_column=col,
                           end_row=6,   end_column=col)
            cell = ws.cell(row=4, column=col)
            cell.value     = SINGLE_COL_HEADERS[col]
            cell.font      = hdr_font
            cell.alignment = hdr_align
            apply_border(ws, f"{cl}4:{cl}6")
        else:
            # cols 7-8: merge horizontally on row 4, then Low/High on row 5, Min/Max on row 6
            if col == 7:
                ws.merge_cells(start_row=4, start_column=7, end_row=4, end_column=8)
                ws.cell(row=4, column=7).value     = "RANGE"
                ws.cell(row=4, column=7).font      = hdr_font
                ws.cell(row=4, column=7).alignment = hdr_align
            if col in ROW5_LABELS:
                ws.cell(row=5, column=col).value = ROW5_LABELS[col]
                ws.cell(row=5, column=col).font  = hdr_font
                ws.cell(row=5, column=col).alignment = hdr_align
            if col in ROW6_LABELS:
                ws.cell(row=6, column=col).value = ROW6_LABELS[col]
                ws.cell(row=6, column=col).font  = hdr_font
                ws.cell(row=6, column=col).alignment = hdr_align
    set_cell1(ws, 'O4', "Coil Side")
    set_cell1(ws, 'O5', "PF Side",           merge_with='O6')

    set_cell1(ws, 'P4', "Relay Board (For DO)")
    set_cell1(ws, 'P5', "Relay Board (For DI)", merge_with='P6')

    set_cell1(ws, 'Q4', "PF Side")
    set_cell1(ws, 'Q5', "Coil Side",          merge_with='Q6')

    # border around full header block
    apply_border(ws, "A4:T6")
    apply_border(ws, "G4:H4")
    apply_border(ws, "G5:G5")
    apply_border(ws, "H5:H5")
    apply_border(ws, "G6:G6")
    apply_border(ws, "H6:H6")
    apply_border(ws, "O4:O4")
    apply_border(ws, "O5:O6")
    apply_border(ws, "P4:P4")
    apply_border(ws, "P5:P6")
    apply_border(ws, "Q4:Q4")
    apply_border(ws, "Q5:Q6")

    sl_no = 1

    while True:
        try:
            n_f = int(input('Enter First Node Number: '))
            n_l = int(input('Enter Last Node Number: '))
            # Check negative
            if n_f < 0:
                print("First node number cannot be negative.")
                continue
            # Check last node
            if n_l < n_f:
                print("Last node number cannot be less than first node number.")
                continue
            break
        except ValueError:
            print("Invalid input. Please enter integers only.")

    while n_f <= n_l:
      node = Node(n_f)
      n_f += 1

    print(f"  Sheet '{sheet_name}' created.")
    return ws


# ==============================================================
#  CoverPage builder
# ==============================================================

def build_coverpage(wb, logo):
    ws = wb.active
    ws.title = "CoverPage"
    ws.sheet_view.showGridLines = False

    # uniform column width & row height
    for col in range(1, 41):
        ws.column_dimensions[get_column_letter(col)].width = 5.7
    for row in range(1, 50):
        ws.row_dimensions[row].height = 25

    # logo
    ws.add_image(logo, "B2")
    logo.width, logo.height = 84, 69

    # ---- row 6: special merges ----
    ws.merge_cells('C6:T6')
    ws.merge_cells('U6:W6')
    ws.merge_cells('X6:AI6')
    ws.merge_cells('AJ6:AK6')
    ws.merge_cells('AM6:AN6')

    # ---- rows 7-16: three-column block merges ----
    for row in range(7, 17):
        merge(ws, 3,  20, row)   # C–T  (label)
        merge(ws, 21, 23, row)   # U–W  (colon)
        merge(ws, 24, 40, row)   # X–AN (value)

    # ---- style rows 6-16 (single loop) ----
    ws['AJ6'].font      = CAL_18B; ws['AJ6'].alignment = RIGHT
    ws['AL6'].font      = CAL_18B; ws['AL6'].alignment = CENTER_H
    ws['AM6'].font      = CAL_16;  ws['AM6'].alignment = LEFT

    for row in range(6, 17):
        set_cell(ws, row, 3,  font=CAL_18B, align=RIGHT)
        set_cell(ws, row, 21, font=CAL_16B, align=CENTER_H)
        ws[f'X{row}'].font      = CAL_16
        ws[f'X{row}'].alignment = LEFT
        apply_border(ws, f"C{row}:AN{row}")

    apply_border(ws, "AJ6:AN6")

    # ---- revision table (rows 23-28) ----
    blocks = [(3,6),(7,14),(15,22),(23,28),(29,34),(35,40)]
    border_ranges = ["C","G","O","W","AC","AI"]
    border_ends   = ["F","N","V","AB","AH","AN"]

    for row in range(23, 29):
        for sc, ec in blocks:
            ws.merge_cells(
                f'{get_column_letter(sc)}{row}:{get_column_letter(ec)}{row}')
        for s, e in zip(border_ranges, border_ends):
            apply_border(ws, f"{s}{row}:{e}{row}")

    apply_border(ws, "C23:AN28", side_style='thick')

    # ---- left-side labels (col C, rows 6-16) ----
    label_map = {
        6:  "DWG/DOC TITLE",
        7:  "PROJECT NAME",
        8:  "CLIENT",
        9:  "END USER",
        10: "VENDOR",
        11: "PO REFERENCE",
        12: "JC NO",
        13: "PACKAGE NAME",
        14: "REFERENCE CLIENT DOC. NAME / NO",
        15: "REFERENCE VENDOR DOCUMENT NO",
        16: "CUSTOMER PROJECT CODE",
    }
    for row, text in label_map.items():
        ws[f'C{row}'] = text

    # ---- colon column ----
    for row in range(6, 17):
        ws[f'U{row}'] = ":"

    # ---- static values ----
    ws['X6']  = "IO List"
    ws['AJ6'] = "REV."
    ws['AL6'] = ":"
    ws['AM6'] = "R0"
    ws['X10'] = "M/s. Subtleweigh Electric (I) Pvt. Ltd."

    # ---- user inputs ----
    global Client_Name, Project_Name, po_reference_no, jc_no

    inputs = [
        ("X7",  "Enter Project Name",         "upper", ""),
        ("X8",  "Enter Client Name",          "upper", "M/s. "),
        ("X9",  "Enter End User Name",        "upper", "M/s. "),
        ("X11", "Enter PO Reference No.",     "upper", ""),
        ("X12", "Enter JC No.",               "upper", ""),
        ("X13", "Enter Package Name",         "upper", ""),
        ("X14", "Enter Reference Client Doc", "upper", ""),
        ("X15", "Enter Reference Vendor Doc", "upper", ""),
        ("X16", "Enter Customer Proj. Code",  "upper", ""),
    ]

    for cell, prompt, case, prefix in inputs:
        #raw = input(f"{prompt}: ").strip()
        raw = widgets.Text(description=f"{prompt}: ")
        val = (raw or "N/A").upper() if case == "upper" else (raw or "N/A").title()
        ws[cell] = f"{prefix}{val}"
        if cell == "X7":  Project_Name    = val
        if cell == "X9":  Client_Name     = val
        if cell == "X11": po_reference_no = val
        if cell == "X12": jc_no           = val

    # ---- revision table content ----
    rev_data = {
        'C28': "REV. NO.", 'G28': "DATE",          'O28': "DESCRIPTION ",
        'W28': "PREPARED BY", 'AC28': "CHECKED BY", 'AI28': "APPROVED BY",
        'C27': "R0",       'G27': today_date_only, 'O27': "FOR APPROVAL",
        'W27': "SP",       'AC27': "SR",            'AI27': "PKR",
    }
    for addr, val in rev_data.items():
        ws[addr] = val

    rev_cols = ['C', 'G', 'O', 'W', 'AC', 'AI']
    for row, size, bold in [(28, 18, True), (27, 16, False)]:
        for col in rev_cols:
            ws[f'{col}{row}'].font      = Font(name='Calibri', size=size, bold=bold)
            ws[f'{col}{row}'].alignment = CENTER_H


# ==============================================================
#  Generate Excel
# ==============================================================
def generate_excel(b):
    # Upload logo
    uploaded  = files.upload()
    logo_file = list(uploaded.keys())[0]
    logo      = Image(logo_file)

    # Build workbook
    wb = Workbook()
    build_coverpage(wb, logo)

    # IO List sheets
    try:
        qty = max(1, int(input("\nHow many IO List sheets? (1, 2, 3...): ")))
    except ValueError:
        print("Invalid – defaulting to 1.")
        qty = 1

    for i in range(1, qty + 1):
        print(f"\n--- Sheet {i} of {qty} ---")
        add_new_IO_List(wb)

    # Save
    file_name = f"{today_date}_IO_List.xlsx"
    wb.save(file_name)
    files.download(file_name)
    print(f"\n✅  Saved: {file_name}")

# ==============================================================
#  Main
# ==============================================================
btn = widgets.Button(
    description="Run Function",
    button_style='success'   # optional: 'success', 'info', 'warning', 'danger'
)

# Link button to function
btn.on_click(generate_excel)

# Display button
display(btn)

Button(button_style='success', description='Run Function', style=ButtonStyle())

Saving logo.png to logo.png


AttributeError: 'Text' object has no attribute 'upper'

In [ ]:
#================  START  : Download from collab  ================
from google.colab import files
files.download(file_name)
#================   END   : Download from collab  ================


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>